# 05. Pengujian Model (Model Testing)
### Implementasi Aspect-Based Sentiment Analysis (ABSA) Berbasis IndoBERT untuk Pemetaan Keluhan Pada Ulasan Produk Monitor

**Referensi metodologi:** Bab III, Sub-bab **1.1.1.3 Pengujian Model** (mencakup 1.1.1.3.1 Prosedur & Skenario Pengujian, dan 1.1.1.3.2 Metrik Evaluasi Kinerja), proposal skripsi.

Notebook ini adalah kelanjutan dari `04_training.ipynb` dan **baru pertama kali membuka `data/splits/test.csv`** — data uji 10% yang belum pernah dilihat model selama pelatihan maupun validasi.

Cakupan notebook ini:

1. Memuat 4 model hasil fine-tuning terbaik (`indobert_acd`, `indobert_asc`, `mbert_acd`, `mbert_asc`) beserta threshold ACD optimal dari `04_training.ipynb`.
2. Pengujian **ACD** pada kedua model.
3. Pengujian **ASC — Skenario 1** (ground-truth aspect) pada kedua model.
4. Pengujian **ASC — Skenario 2** (end-to-end dari hasil ACD) pada kedua model.
5. Confusion Matrix per kelas aspek/sentimen.
6. Interval kepercayaan (Confidence Interval) via bootstrapping.
7. Uji McNemar (IndoBERT vs mBERT).

> **Catatan cakupan:** Notebook ini **berhenti pada penyajian metrik dan artefak pengujian** (angka, tabel, confusion matrix, hasil uji statistik). Interpretasi/analisis atas hasil-hasil ini (mis. pembacaan pola kesalahan, pembahasan signifikansi McNemar terhadap rumusan masalah) adalah bagian dari **1.1.1.4 Analisis dan Implementasi**, yang akan dibahas secara terpisah — bukan bagian dari notebook implementasi ini.

## 0. Persiapan Lingkungan

In [ ]:
# !pip install -q transformers torch scikit-learn statsmodels pandas matplotlib seaborn

In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss, confusion_matrix, classification_report
)
from statsmodels.stats.contingency_tables import mcnemar

from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Torch version :", torch.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device dipakai:", DEVICE)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)

In [ ]:
BASE_DIR  = Path("..")
DATA_DIR  = BASE_DIR / "data" / "splits"
MODEL_DIR = BASE_DIR / "model"
EVAL_DIR  = MODEL_DIR / "evaluation"
FIG_DIR   = EVAL_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

TEST_PATH = DATA_DIR / "test.csv"   # baru dibuka di notebook ini

# Artefak dari 04_training.ipynb
with open(MODEL_DIR / "label_maps.json", encoding="utf-8") as f:
    label_maps = json.load(f)
with open(MODEL_DIR / "training_summary.json", encoding="utf-8") as f:
    training_summary = json.load(f)
with open(MODEL_DIR / "acd_optimal_thresholds.json", encoding="utf-8") as f:
    acd_thresholds = json.load(f)

ASPECT_LABELS    = label_maps["aspect_labels"]
SENTIMENT_LABELS = label_maps["sentiment_labels"]
aspect2id    = label_maps["aspect2id"]
sentiment2id = label_maps["sentiment2id"]

MODEL_KEYS = ["indobert", "mbert"]

print("Aspek   :", ASPECT_LABELS)
print("Sentimen:", SENTIMENT_LABELS)
print("Threshold ACD optimal:", acd_thresholds)

## 1. Memuat Data Uji

Sesuai Batasan Masalah poin 4 dan Sub-bab 1.1.1.3, evaluasi akhir dilakukan pada data uji sebesar 10% yang disisihkan sejak `03_data_preparation.ipynb` dan **tidak pernah dilihat model** selama pelatihan/validasi.

In [ ]:
df_test = pd.read_csv(TEST_PATH)
df_test["Aspek"] = df_test["Aspek"].astype(str).str.strip().str.upper()
df_test["Sentimen"] = df_test["Sentimen"].astype(str).str.strip().str.capitalize()

print(f"Baris data uji (long format): {len(df_test)}  | ulasan unik: {df_test['Ulasan'].nunique()}")
df_test.head()

### 1.1 Data Uji untuk ACD (agregasi per ulasan unik)

Mengikuti prosedur yang identik dengan penyiapan data latih pada `04_training.ipynb`, agar representasi label konsisten.

In [ ]:
def build_acd_test_df(df_long: pd.DataFrame) -> pd.DataFrame:
    grouped = (
        df_long.groupby("Ulasan")["Aspek"]
        .apply(lambda s: sorted(set(s)))
        .reset_index()
        .rename(columns={"Aspek": "aspect_set"})
    )
    return grouped

acd_test_df = build_acd_test_df(df_test)

def to_multihot(aspect_sets, aspect_labels):
    idx = {a: i for i, a in enumerate(aspect_labels)}
    mat = np.zeros((len(aspect_sets), len(aspect_labels)), dtype=int)
    for r, s in enumerate(aspect_sets):
        for a in s:
            mat[r, idx[a]] = 1
    return mat

acd_test_labels = to_multihot(acd_test_df["aspect_set"], ASPECT_LABELS)
print("Jumlah ulasan uji (ACD):", len(acd_test_df))
acd_test_df.head()

### 1.2 Data Uji untuk ASC — Skenario 1 (Ground-Truth Aspect)

Data long-format sudah langsung sesuai untuk skenario ini: setiap baris adalah satu instance (ulasan, aspek-benar, sentimen-benar).

In [ ]:
asc_test_df = df_test[["Ulasan", "Aspek", "Sentimen"]].copy()
asc_test_df["label"] = asc_test_df["Sentimen"].map(sentiment2id)
assert asc_test_df["label"].isna().sum() == 0

print("Jumlah instance ASC uji (skenario ground-truth):", len(asc_test_df))
asc_test_df.head()

## 2. Memuat Model Hasil Fine-Tuning

Keempat model dimuat langsung dari direktori `model/{model}_{task}/best_model/` yang disimpan `04_training.ipynb`, bersama tokenizer dan `max_length` yang tercatat di `training_summary.json` — memastikan preprocessing saat pengujian identik dengan saat pelatihan.

In [ ]:
def load_finetuned(model_key: str, task: str):
    info = training_summary[f"{model_key}_{task}"]
    path = info["saved_path"]
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path).to(DEVICE)
    model.eval()
    return model, tokenizer, info["max_length"]

loaded = {}
for model_key in MODEL_KEYS:
    for task in ["acd", "asc"]:
        model, tokenizer, max_len = load_finetuned(model_key, task)
        loaded[(model_key, task)] = {"model": model, "tokenizer": tokenizer, "max_length": max_len}
        print(f"Dimuat: {model_key}_{task} (max_length={max_len})")

### 2.1 Fungsi Inferensi Batch

In [ ]:
@torch.no_grad()
def predict_acd(model, tokenizer, texts, max_length, threshold, batch_size=32):
    model.eval()
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i + batch_size])
        enc = tokenizer(batch, truncation=True, max_length=max_length, padding=True, return_tensors="pt").to(DEVICE)
        logits = model(**enc).logits
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
    probs = np.vstack(all_probs)
    preds = (probs >= threshold).astype(int)
    return probs, preds


@torch.no_grad()
def predict_asc(model, tokenizer, texts_a, texts_b, max_length, batch_size=32):
    model.eval()
    all_logits = []
    for i in range(0, len(texts_a), batch_size):
        ba = list(texts_a[i:i + batch_size])
        bb = list(texts_b[i:i + batch_size])
        enc = tokenizer(ba, bb, truncation=True, max_length=max_length, padding=True, return_tensors="pt").to(DEVICE)
        logits = model(**enc).logits
        all_logits.append(logits.cpu().numpy())
    logits = np.vstack(all_logits)
    preds = np.argmax(logits, axis=1)
    return logits, preds

## 3. Pengujian ACD (Aspect Category Detection)

Threshold yang dipakai adalah threshold optimal **per model** hasil kalibrasi pada validation set (`04_training.ipynb`, bagian 8) — bukan 0.5 tetap — sesuai penjelasan proposal.

In [ ]:
acd_results = {}
acd_preds_store = {}   # dipakai kembali di ASC Skenario 2

for model_key in MODEL_KEYS:
    entry = loaded[(model_key, "acd")]
    threshold = acd_thresholds[model_key]["threshold"]

    probs, preds = predict_acd(
        entry["model"], entry["tokenizer"],
        acd_test_df["Ulasan"].tolist(), entry["max_length"], threshold
    )
    acd_preds_store[model_key] = preds

    metrics = {
        "accuracy_subset": accuracy_score(acd_test_labels, preds),  # exact-match accuracy (ketat, multi-label)
        "precision_macro": precision_score(acd_test_labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(acd_test_labels, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(acd_test_labels, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(acd_test_labels, preds, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(acd_test_labels, preds),
        "threshold_used": threshold,
    }
    acd_results[model_key] = metrics
    print(f"\n=== ACD - {model_key} ===")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

acd_results_df = pd.DataFrame(acd_results).T
acd_results_df.to_csv(EVAL_DIR / "acd_test_metrics.csv")
acd_results_df

### 3.1 Confusion Matrix per Kelas Aspek (ACD)

Untuk setiap kategori aspek, dibentuk confusion matrix biner (Ada/Tidak Ada) sesuai penjelasan proposal Sub-bab 1.1.1.3.2.

In [ ]:
def plot_acd_confusion_grid(y_true, y_pred, aspect_labels, title, save_path=None):
    n = len(aspect_labels)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.6 * nrows))
    axes = np.array(axes).flatten()

    for i, aspek in enumerate(aspect_labels):
        cm = confusion_matrix(y_true[:, i], y_pred[:, i], labels=[0, 1])
        ax = axes[i]
        ax.imshow(cm, cmap="Blues")
        ax.set_title(f"Aspek {aspek}", fontsize=11, fontweight="bold")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["Tidak Ada", "Ada"], fontsize=8)
        ax.set_yticks([0, 1]); ax.set_yticklabels(["Tidak Ada", "Ada"], fontsize=8)
        for r in range(2):
            for c in range(2):
                ax.text(c, r, cm[r, c], ha="center", va="center",
                         color="white" if cm[r, c] > cm.max() / 2 else "black", fontsize=10)
    for j in range(n, len(axes)):
        axes[j].axis("off")

    fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

for model_key in MODEL_KEYS:
    plot_acd_confusion_grid(
        acd_test_labels, acd_preds_store[model_key], ASPECT_LABELS,
        f"Confusion Matrix per Aspek — ACD ({model_key})",
        save_path=FIG_DIR / f"acd_confusion_{model_key}.png"
    )

## 4. Pengujian ASC — Skenario 1 (Ground-Truth Aspect)

Sesuai Sub-bab 1.1.1.3.1: kategori aspek yang dimasukkan ke model adalah **label yang sudah dipastikan kebenarannya**, sehingga kesalahan ACD tidak memengaruhi skor sentimen pada skenario ini — mengukur kemampuan murni model dalam memahami polaritas.

In [ ]:
asc_s1_results = {}
asc_s1_preds_store = {}

for model_key in MODEL_KEYS:
    entry = loaded[(model_key, "asc")]
    logits, preds = predict_asc(
        entry["model"], entry["tokenizer"],
        asc_test_df["Ulasan"].tolist(), asc_test_df["Aspek"].tolist(), entry["max_length"]
    )
    asc_s1_preds_store[model_key] = preds
    y_true = asc_test_df["label"].values

    metrics = {
        "accuracy": accuracy_score(y_true, preds),
        "precision_macro": precision_score(y_true, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(y_true, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, preds, average="macro", zero_division=0),
    }
    asc_s1_results[model_key] = metrics
    print(f"\n=== ASC Skenario 1 (ground-truth aspect) - {model_key} ===")
    print(classification_report(y_true, preds, target_names=SENTIMENT_LABELS, zero_division=0))

asc_s1_df = pd.DataFrame(asc_s1_results).T
asc_s1_df.to_csv(EVAL_DIR / "asc_scenario1_test_metrics.csv")
asc_s1_df

### 4.1 Confusion Matrix Sentimen (Keseluruhan) dan per Aspek

Confusion matrix 3×3 keseluruhan ditampilkan langsung di notebook; versi rincian per kategori aspek (9 confusion matrix per model) disimpan sebagai berkas gambar di `model/evaluation/figures/` agar notebook tetap ringkas dan mudah dibaca — dapat dibuka satu per satu saat pembahasan pada tahap Analisis.

In [ ]:
def plot_sentiment_cm(y_true, y_pred, labels, title, save_path=None):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
    fig, ax = plt.subplots(figsize=(4.8, 4.3))
    ax.imshow(cm, cmap="Purples")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual")
    ax.set_title(title, fontsize=11, fontweight="bold")
    for r in range(len(labels)):
        for c in range(len(labels)):
            ax.text(c, r, cm[r, c], ha="center", va="center",
                     color="white" if cm[r, c] > cm.max() / 2 else "black")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return cm

for model_key in MODEL_KEYS:
    plot_sentiment_cm(
        asc_test_df["label"].values, asc_s1_preds_store[model_key], SENTIMENT_LABELS,
        f"Confusion Matrix Sentimen (Keseluruhan) — ASC Skenario 1 ({model_key})",
        save_path=FIG_DIR / f"asc_s1_confusion_overall_{model_key}.png"
    )

In [ ]:
# Confusion matrix per aspek (disimpan ke file, tidak ditampilkan satu-satu agar notebook ringkas)
per_aspect_f1_rows = []

for model_key in MODEL_KEYS:
    y_true_all = asc_test_df["label"].values
    y_pred_all = asc_s1_preds_store[model_key]

    for aspek in ASPECT_LABELS:
        mask = (asc_test_df["Aspek"].values == aspek)
        if mask.sum() == 0:
            continue
        yt, yp = y_true_all[mask], y_pred_all[mask]

        cm = confusion_matrix(yt, yp, labels=list(range(len(SENTIMENT_LABELS))))
        fig, ax = plt.subplots(figsize=(3.6, 3.2))
        ax.imshow(cm, cmap="Purples")
        ax.set_xticks(range(len(SENTIMENT_LABELS))); ax.set_xticklabels(SENTIMENT_LABELS, rotation=30, ha="right", fontsize=8)
        ax.set_yticks(range(len(SENTIMENT_LABELS))); ax.set_yticklabels(SENTIMENT_LABELS, fontsize=8)
        ax.set_title(f"{model_key} — Aspek {aspek}", fontsize=10)
        for r in range(len(SENTIMENT_LABELS)):
            for c in range(len(SENTIMENT_LABELS)):
                ax.text(c, r, cm[r, c], ha="center", va="center", fontsize=8,
                         color="white" if cm[r, c] > cm.max() / 2 else "black")
        plt.tight_layout()
        fig.savefig(FIG_DIR / f"asc_s1_confusion_{model_key}_{aspek}.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

        per_aspect_f1_rows.append({
            "model": model_key, "aspek": aspek, "n_instance": int(mask.sum()),
            "f1_macro": f1_score(yt, yp, average="macro", zero_division=0),
            "accuracy": accuracy_score(yt, yp),
        })

per_aspect_f1_df = pd.DataFrame(per_aspect_f1_rows)
per_aspect_f1_df.to_csv(EVAL_DIR / "asc_scenario1_per_aspect_metrics.csv", index=False)
per_aspect_f1_df.pivot(index="aspek", columns="model", values="f1_macro")

## 5. Pengujian ASC — Skenario 2 (End-to-End dari Hasil ACD)

Sesuai Sub-bab 1.1.1.3.1, skenario ini memakai **hasil prediksi ACD** (bukan ground-truth) sebagai input aspek ke model ASC — mengukur performa sistem end-to-end termasuk *cascading error*.

**Skema evaluasi:** Karena himpunan aspek prediksi dapat berbeda dari himpunan aspek sebenarnya (aspek meleset dari ACD tidak akan pernah dinilai sentimennya; aspek yang salah dideteksi tetap diberi prediksi sentimen), evaluasi dilakukan pada level **pasangan (aspek, sentimen)** — setiap ulasan direpresentasikan sebagai vektor multi-hot berdimensi 9×3=27 (9 aspek × 3 sentimen), sama seperti pendekatan evaluasi pasangan aspek-polaritas pada tugas ABSA gabungan (Pontiki et al., 2014). Sebuah pasangan dihitung benar hanya jika **aspek terdeteksi dan sentimennya tepat sekaligus**, sehingga kesalahan pada tahap ACD otomatis tercermin pada skor akhir skenario ini.

In [ ]:
PAIR_UNIVERSE = [(a, s) for a in ASPECT_LABELS for s in SENTIMENT_LABELS]
pair2idx = {p: i for i, p in enumerate(PAIR_UNIVERSE)}

def true_pair_matrix(df_long_test, aspect_test_df):
    """Vektor multi-hot 27-dim (aspek,sentimen) BENAR, satu baris per ulasan unik
    (urutan mengikuti aspect_test_df)."""
    truth_map = (
        df_long_test.groupby("Ulasan")
        .apply(lambda g: dict(zip(g["Aspek"], g["Sentimen"])))
    )
    mat = np.zeros((len(aspect_test_df), len(PAIR_UNIVERSE)), dtype=int)
    for r, ulasan in enumerate(aspect_test_df["Ulasan"]):
        for a, s in truth_map[ulasan].items():
            mat[r, pair2idx[(a, s)]] = 1
    return mat

y_true_pairs = true_pair_matrix(df_test, acd_test_df)
print("Bentuk matriks pasangan ground-truth:", y_true_pairs.shape)

In [ ]:
asc_s2_results = {}
asc_s2_pred_pairs_store = {}

for model_key in MODEL_KEYS:
    acd_pred = acd_preds_store[model_key]          # (n_ulasan, 9) biner, hasil bagian 3
    asc_entry = loaded[(model_key, "asc")]

    # Membangun daftar (index_ulasan, ulasan, aspek_prediksi) utk semua aspek yg diprediksi ACD
    query_rows = []
    for r, ulasan in enumerate(acd_test_df["Ulasan"]):
        predicted_aspects = [ASPECT_LABELS[i] for i in range(len(ASPECT_LABELS)) if acd_pred[r, i] == 1]
        for a in predicted_aspects:
            query_rows.append((r, ulasan, a))

    pred_matrix = np.zeros((len(acd_test_df), len(PAIR_UNIVERSE)), dtype=int)

    if len(query_rows) > 0:
        texts_a = [q[1] for q in query_rows]
        texts_b = [q[2] for q in query_rows]
        _, sent_preds = predict_asc(asc_entry["model"], asc_entry["tokenizer"], texts_a, texts_b, asc_entry["max_length"])

        for (r, ulasan, a), s_idx in zip(query_rows, sent_preds):
            s = SENTIMENT_LABELS[s_idx]
            pred_matrix[r, pair2idx[(a, s)]] = 1

    asc_s2_pred_pairs_store[model_key] = pred_matrix

    metrics = {
        "accuracy_subset": accuracy_score(y_true_pairs, pred_matrix),
        "precision_macro": precision_score(y_true_pairs, pred_matrix, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true_pairs, pred_matrix, average="macro", zero_division=0),
        "f1_micro": f1_score(y_true_pairs, pred_matrix, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true_pairs, pred_matrix, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(y_true_pairs, pred_matrix),
    }
    asc_s2_results[model_key] = metrics
    print(f"\n=== ASC Skenario 2 (end-to-end dari ACD) - {model_key} ===")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

asc_s2_df = pd.DataFrame(asc_s2_results).T
asc_s2_df.to_csv(EVAL_DIR / "asc_scenario2_test_metrics.csv")
asc_s2_df

### 5.1 Perbandingan Skenario 1 vs Skenario 2

Selisih Macro-F1 antara kedua skenario mengindikasikan besarnya dampak *cascading error* dari tahap ACD terhadap performa sentimen end-to-end.

In [ ]:
comparison_df = pd.DataFrame({
    "f1_macro_skenario1_ground_truth": {m: asc_s1_results[m]["f1_macro"] for m in MODEL_KEYS},
    "f1_macro_skenario2_end_to_end": {m: asc_s2_results[m]["f1_macro"] for m in MODEL_KEYS},
})
comparison_df["selisih (cascading effect)"] = (
    comparison_df["f1_macro_skenario1_ground_truth"] - comparison_df["f1_macro_skenario2_end_to_end"]
)
comparison_df.to_csv(EVAL_DIR / "asc_scenario_comparison.csv")
comparison_df

## 6. Interval Kepercayaan (Confidence Interval) via Bootstrapping

Mengikuti proposal Sub-bab 1.1.1.3.2 (merujuk Mokhtar et al., 2023), estimasi rentang kinerja dilakukan dengan resampling berulang (bootstrapping) pada data uji, dihitung untuk metrik **Macro-F1** pada tiap tugas/skenario dan tiap model.

In [ ]:
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=1000, ci=95, seed=SEED):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    scores = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        scores[b] = metric_fn(y_true[idx], y_pred[idx])
    lower = np.percentile(scores, (100 - ci) / 2)
    upper = np.percentile(scores, 100 - (100 - ci) / 2)
    return float(np.mean(scores)), float(lower), float(upper)

f1_macro_fn = lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0)

ci_rows = []
for model_key in MODEL_KEYS:
    mean, lo, hi = bootstrap_ci(acd_test_labels, acd_preds_store[model_key], f1_macro_fn)
    ci_rows.append({"model": model_key, "task_scenario": "ACD", "f1_macro_mean": mean, "ci95_lower": lo, "ci95_upper": hi})

    mean, lo, hi = bootstrap_ci(asc_test_df["label"].values, asc_s1_preds_store[model_key], f1_macro_fn)
    ci_rows.append({"model": model_key, "task_scenario": "ASC_skenario1", "f1_macro_mean": mean, "ci95_lower": lo, "ci95_upper": hi})

    mean, lo, hi = bootstrap_ci(y_true_pairs, asc_s2_pred_pairs_store[model_key], f1_macro_fn)
    ci_rows.append({"model": model_key, "task_scenario": "ASC_skenario2", "f1_macro_mean": mean, "ci95_lower": lo, "ci95_upper": hi})

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(EVAL_DIR / "bootstrap_confidence_intervals.csv", index=False)
ci_df

## 7. Uji McNemar (IndoBERT vs mBERT)

Sesuai proposal Sub-bab 1.1.1.3.2 (merujuk Syazal & Yulianti, 2025), uji McNemar dilakukan berpasangan pada data uji yang sama untuk menentukan apakah selisih performa IndoBERT terhadap mBERT signifikan secara statistik. Uji dilakukan pada indikator benar/salah per instance:

- **ACD**: per kategori aspek (indikator benar/salah biner tiap aspek), ditambah versi *exact-match* seluruh label per ulasan.
- **ASC Skenario 1**: indikator benar/salah klasifikasi sentimen per instance (skenario ini dipakai karena murni mengukur kemampuan sentimen tanpa bias kesalahan ACD).

In [ ]:
def run_mcnemar(correct_a, correct_b):
    correct_a = np.asarray(correct_a, dtype=bool)
    correct_b = np.asarray(correct_b, dtype=bool)
    both_correct = int(np.sum(correct_a & correct_b))
    only_a = int(np.sum(correct_a & ~correct_b))
    only_b = int(np.sum(~correct_a & correct_b))
    both_wrong = int(np.sum(~correct_a & ~correct_b))
    table = [[both_correct, only_a], [only_b, both_wrong]]
    result = mcnemar(table, exact=(min(only_a, only_b) < 25), correction=True)
    return table, float(result.statistic), float(result.pvalue)

mcnemar_rows = []

# --- ACD: exact-match per ulasan ---
correct_indobert = np.all(acd_test_labels == acd_preds_store["indobert"], axis=1)
correct_mbert = np.all(acd_test_labels == acd_preds_store["mbert"], axis=1)
table, stat, p = run_mcnemar(correct_indobert, correct_mbert)
mcnemar_rows.append({"pengujian": "ACD_exact_match_seluruh_label", "aspek": "-",
                      "statistic": stat, "p_value": p, "contingency_table": table})

# --- ACD: per aspek ---
for i, aspek in enumerate(ASPECT_LABELS):
    correct_indobert = (acd_test_labels[:, i] == acd_preds_store["indobert"][:, i])
    correct_mbert = (acd_test_labels[:, i] == acd_preds_store["mbert"][:, i])
    table, stat, p = run_mcnemar(correct_indobert, correct_mbert)
    mcnemar_rows.append({"pengujian": "ACD_per_aspek", "aspek": aspek,
                          "statistic": stat, "p_value": p, "contingency_table": table})

# --- ASC Skenario 1: per instance ---
y_true_asc = asc_test_df["label"].values
correct_indobert = (y_true_asc == asc_s1_preds_store["indobert"])
correct_mbert = (y_true_asc == asc_s1_preds_store["mbert"])
table, stat, p = run_mcnemar(correct_indobert, correct_mbert)
mcnemar_rows.append({"pengujian": "ASC_skenario1_ground_truth", "aspek": "-",
                      "statistic": stat, "p_value": p, "contingency_table": table})

mcnemar_df = pd.DataFrame(mcnemar_rows)
mcnemar_df.to_csv(EVAL_DIR / "mcnemar_test_results.csv", index=False)
mcnemar_df

## 8. Ringkasan Artefak Pengujian

Seluruh keluaran tahap ini tersimpan di `model/evaluation/`:

| Berkas | Isi |
|---|---|
| `acd_test_metrics.csv` | Accuracy, Precision, Recall, Micro/Macro-F1, Hamming Loss — ACD, kedua model |
| `asc_scenario1_test_metrics.csv` | Metrik ASC skenario ground-truth aspect, kedua model |
| `asc_scenario1_per_aspect_metrics.csv` | Macro-F1 & accuracy ASC skenario 1, dirinci per kategori aspek |
| `asc_scenario2_test_metrics.csv` | Metrik ASC skenario end-to-end (dari hasil ACD), kedua model |
| `asc_scenario_comparison.csv` | Selisih Macro-F1 skenario 1 vs 2 (indikasi *cascading error*) |
| `bootstrap_confidence_intervals.csv` | CI 95% Macro-F1 (ACD, ASC skenario 1 & 2), bootstrapping 1000 iterasi |
| `mcnemar_test_results.csv` | Statistik & p-value uji McNemar (ACD per aspek + exact-match, ASC skenario 1) |
| `figures/acd_confusion_{model}.png` | Confusion matrix 9 aspek — ACD |
| `figures/asc_s1_confusion_overall_{model}.png` | Confusion matrix sentimen keseluruhan — ASC skenario 1 |
| `figures/asc_s1_confusion_{model}_{aspek}.png` | Confusion matrix sentimen per aspek — ASC skenario 1 |

**Tahap selanjutnya (di luar cakupan notebook ini):** seluruh angka dan artefak di atas menjadi bahan baku untuk **1.1.1.4 Analisis dan Implementasi** — pembacaan pola kesalahan pada confusion matrix, interpretasi signifikansi uji McNemar terhadap Rumusan Masalah, dan pembahasan komparatif IndoBERT vs mBERT — yang akan dibahas secara terpisah (bukan bagian kode notebook ini), baru setelah itu dilanjutkan ke prototipe Streamlit.